# Load Dataset



In [ ]:
import pandas as pd

df = pd.read_csv("/week_3/day_2/Exercises/train.csv")

df.head(), df.info()


# Exercise 1 – Duplicate Detection and Removal

In [ ]:
# Check duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows before removal: {duplicates}")

# Remove duplicates
df = df.drop_duplicates()

print(f"Number of duplicate rows after removal: {df.duplicated().sum()}")
print(f"Shape after duplicate removal: {df.shape}")


# Exercise 2 : Handling Missing Values


In [ ]:

# Check missing values
print("Missing values before handling:")
print(df.isnull().sum())


# Drop Cabin (too many missing values)
df = df.drop(columns=["Cabin"])

# Fill Age with median
df["Age"].fillna(df["Age"].median(), inplace=True)

# Fill Embarked with mode 
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)

print("\nMissing values after handling:")
print(df.isnull().sum())


# Exercise 3 : Feature Engineering


In [ ]:

# Create FamilySize
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Extract Title from Name
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Simplify rare titles
df["Title"] = df["Title"].replace(
    ['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'],
    'Rare'
)
df["Title"] = df["Title"].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

# Encode categorical features
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

df["Sex"] = le.fit_transform(df["Sex"])
df["Embarked"] = le.fit_transform(df["Embarked"])
df["Title"] = le.fit_transform(df["Title"])

df.head()


# Exercise 4 : Outlier Detection and Handling


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot before handling
plt.figure(figsize=(10,4))
sns.boxplot(data=df[["Age", "Fare"]])
plt.title("Before Outlier Treatment")
plt.show()

# IQR method for Fare
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
df["Fare"] = np.where(df["Fare"] > upper, upper, df["Fare"])
df["Fare"] = np.where(df["Fare"] < lower, lower, df["Fare"])

# Age: optional capping at 99th percentile
cap_age = df["Age"].quantile(0.99)
df["Age"] = np.where(df["Age"] > cap_age, cap_age, df["Age"])

# Boxplot after handling
plt.figure(figsize=(10,4))
sns.boxplot(data=df[["Age", "Fare"]])
plt.title("After Outlier Treatment")
plt.show()


# Exercise 5 : Data Standardization and Normalization


In [ ]:

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Selecting numerical columns
num_cols = ["Age", "Fare", "FamilySize"]

# Standardization (mean=0, std=1)
scaler_std = StandardScaler()
df_std = df.copy()
df_std[num_cols] = scaler_std.fit_transform(df[num_cols])

# Normalization (0-1)
scaler_mm = MinMaxScaler()
df_mm = df.copy()
df_mm[num_cols] = scaler_mm.fit_transform(df[num_cols])

print("Standardized sample:")
display(df_std[num_cols].head())

print("Normalized sample:")
display(df_mm[num_cols].head())


# Exercise 6 : Feature Encoding


In [ ]:

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, columns=["Embarked", "Title"], drop_first=True)

df_encoded.head()


# Exercise 7 : Data Transformation for Age Feature


In [ ]:

# Defining bins and labels
bins = [0, 12, 18, 60, 100]
labels = ['Child', 'Teen', 'Adult', 'Senior']

# Cutting into bins
df["AgeGroup"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)

# One-hot encode AgeGroup
df = pd.get_dummies(df, columns=["AgeGroup"], drop_first=True)


df[["Age", "AgeGroup_Teen", "AgeGroup_Adult", "AgeGroup_Senior"]].head()
